## Exploration Notebook

### Setup

In [ ]:
import pandas as pd  # noqa
from scipy import stats  # noqa
import seaborn as sns  # noqa
from utils.soda3_client import SOQL_Querying  # noqa
from dotenv import load_dotenv
import os
import geopandas

In [2]:
load_dotenv()

True

## Pre Expoloration Setup

### Stop Data Engineering

In [3]:
stop_data = SOQL_Querying(dataset_url="https://data.ny.gov/api/v3/views/2ucp-7wg5/query.json", app_token=os.environ["SOCRATA_DEV"])
stop_ridership_data = SOQL_Querying(dataset_url="https://data.ny.gov/api/v3/views/fvdm-uavx/query.json", app_token=os.environ["SOCRATA_DEV"])

In [4]:
stop_data.query(soql_query="""
    select distinct route_id
    where route_id like '%B82%'    
    """)

,route_id
0,B82
1,B82
2,B82
3,B82
4,B82
...,...
995,B82+
996,B82+
997,B82+
998,B82


In [5]:
stop_ridership_data.query("""
    select distinct route_id
    where route_id like '%B82%'             
    """)

,route_id
0,B82
1,B82+


We can confirm the routes for the B82 and B82-SBS have the same ID within both datasets.

In [7]:
stop_data.query("""
    select * limit 1
    """)

,valid_from,valid_to,in_effect,route_id,route_short_name,route_long_name,route_description,route_color,stop_id,stop_name,...,direction,revenue_stop,timepoint,boarding,alighting,is_cbd,latitude,longitude,bundle,georeference
0,2021-11-15T00:00:00.000,2021-12-19T00:00:00.000,false,B1,B1,Bay Ridge - Manhattan Beach,via 86th St / Ocean Pkwy,00AEEF,300000,ORIENTAL BL/MACKENZIE ST,...,N,1,1,1,1,false,40.57835,-73.940029,2021Sep,"{'type': 'Point', 'coordinates': [-73.940029, ..."


In [8]:
stop_ridership_data.query("""
    select * limit 1
    """)

,date,hour,route_id,direction,stop_id,stop_sequence,boardings,alightings,trips
0,2024-09-30T00:00:00.000,2024-09-30 12:00:00 AM,B25,E,308025,2,2,0,1


In [12]:
stop_location_df = stop_data.query("""
    select
        *
    where route_id like '%B82%'
    """)

stop_location_gdf = geopandas.GeoDataFrame(
    stop_location_df, geometry=geopandas.points_from_xy(stop_location_df.longitude, stop_location_df.latitude)
)
stop_location_df.head()

,valid_from,valid_to,in_effect,route_id,route_short_name,route_long_name,route_description,route_color,stop_id,stop_name,...,direction,revenue_stop,timepoint,boarding,alighting,is_cbd,latitude,longitude,bundle,georeference
0,2021-11-15T00:00:00.000,2021-12-19T00:00:00.000,false,B82,B82,Coney Island - Spring Creek Towers,via Bay Pkwy / Kings Hwy / Flatlands Av,6CBE45,300418,CROPSEY AV/BAY 49 ST,...,E,1,0,1,1,false,40.586092,-73.987889,2021Sep,"{'type': 'Point', 'coordinates': [-73.987889, ..."
1,2021-11-15T00:00:00.000,2021-12-19T00:00:00.000,false,B82,B82,Coney Island - Spring Creek Towers,via Bay Pkwy / Kings Hwy / Flatlands Av,6CBE45,300419,CROPSEY AV/BAY 46 ST,...,E,1,0,1,1,false,40.588211,-73.989591,2021Sep,"{'type': 'Point', 'coordinates': [-73.989591, ..."
2,2021-11-15T00:00:00.000,2021-12-19T00:00:00.000,false,B82,B82,Coney Island - Spring Creek Towers,via Bay Pkwy / Kings Hwy / Flatlands Av,6CBE45,300420,CROPSEY AV/BAY 44 ST,...,E,1,0,1,1,false,40.58941,-73.990563,2021Sep,"{'type': 'Point', 'coordinates': [-73.990563, ..."
3,2021-11-15T00:00:00.000,2021-12-19T00:00:00.000,false,B82,B82,Coney Island - Spring Creek Towers,via Bay Pkwy / Kings Hwy / Flatlands Av,6CBE45,300421,CROPSEY AV/26 AV,...,E,1,0,1,1,false,40.590742,-73.99164,2021Sep,"{'type': 'Point', 'coordinates': [-73.99164, 4..."
4,2021-11-15T00:00:00.000,2021-12-19T00:00:00.000,false,B82,B82,Coney Island - Spring Creek Towers,via Bay Pkwy / Kings Hwy / Flatlands Av,6CBE45,300422,CROPSEY AV/25 AV,...,E,1,0,1,1,false,40.592279,-73.992878,2021Sep,"{'type': 'Point', 'coordinates': [-73.992878, ..."


In [16]:
stop_ridership_df = stop_ridership_data.query("""
    select
        route_id,
        stop_id, 
        stop_sequence,
        date_extract_m(date) as month_2025,
        sum(boardings),
        sum(alightings)
    where 
        route_id like '%B82%'
        and 
        date_extract_y(date) = 2025 
    group by
        route_id,
        stop_id,
        stop_sequence,
        date_extract_m(date)
    order by
        stop_id,
        month_2025
    """)
stop_ridership_df.head()

,route_id,stop_id,stop_sequence,month_2025,sum_boardings,sum_alightings
0,B82,300418,5,1,2239,563
1,B82,300418,5,2,1935,500
2,B82,300418,5,3,2193,578
3,B82,300418,5,4,2313,589
4,B82,300418,5,5,2377,646


In [17]:
stop_ridership_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   route_id        1000 non-null   str  
 1   stop_id         1000 non-null   str  
 2   stop_sequence   1000 non-null   str  
 3   month_2025      1000 non-null   str  
 4   sum_boardings   1000 non-null   str  
 5   sum_alightings  1000 non-null   str  
dtypes: str(6)
memory usage: 47.0 KB
